<a href="https://colab.research.google.com/github/MrKarahanly/gulstan/blob/main/i_teka.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ИМПОРТЫ
import requests
from bs4 import BeautifulSoup
import pandas as pd
import sqlite3
import re
import time
import random
import os
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns

# Настройка графиков
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 10

print("Библиотеки загружены")

# КЛАСС ПАРСЕРА

class ITekaParser:
    """Парсер сайта i-teka.kz с пагинацией"""

    def __init__(self, city="astana", max_pages=None, delay_range=(1, 2)):
        self.base_url = "https://i-teka.kz"
        self.city = city
        self.max_pages = max_pages  # None = все страницы
        self.delay_range = delay_range
        self.headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.0"
        }
        self.session = requests.Session()
        self.session.headers.update(self.headers)

    def _get_page(self, url, retries=3):
        """Получение страницы с повторными попытками"""
        for attempt in range(retries):
            try:
                response = self.session.get(url, timeout=30)
                response.raise_for_status()
                return response.text
            except requests.exceptions.RequestException as e:
                print(f"Ошибка (попытка {attempt+1}/{retries}): {e}")
                time.sleep(3 * (attempt + 1))
        return None

    def _delay(self):
        """Пауза между запросами"""
        time.sleep(random.uniform(*self.delay_range))

    def _parse_medicines_from_text(self, html):
        """Парсинг препаратов из текста страницы"""
        soup = BeautifulSoup(html, "lxml")
        lines = [line.strip() for line in soup.get_text("\n").split("\n") if line.strip()]

        records = []
        current = {}

        for line in lines:
            # Признаки названия препарата
            if any(x in line.lower() for x in ["таблет", "раствор", "мл", "мг", "капс", "шприц", "мазь", "гель", "капли", "сироп"]):
                # Сохраняем предыдущий, если есть
                if current.get("name") and current.get("pharmacies"):
                    records.append({
                        "name": current["name"],
                        "mnn": current.get("mnn"),
                        "prescription": current.get("prescription"),
                        "pharmacies_count": current["pharmacies"],
                        "city": self.city,
                        "parsed_at": datetime.now().isoformat()
                    })

                current = {"name": line}

            elif line.startswith("МНН:"):
                current["mnn"] = line.replace("МНН:", "").strip()

            elif line in ["Без рецепта", "По рецепту"]:
                current["prescription"] = line

            elif "Найден" in line and "аптек" in line:
                match = re.search(r"(\d+)", line)
                if match:
                    current["pharmacies"] = int(match.group(1))

        # Последняя запись
        if current.get("name") and current.get("pharmacies"):
            records.append({
                "name": current["name"],
                "mnn": current.get("mnn"),
                "prescription": current.get("prescription"),
                "pharmacies_count": current["pharmacies"],
                "city": self.city,
                "parsed_at": datetime.now().isoformat()
            })

        return records

    def _get_total_pages(self, html):
        """Определение количества страниц"""
        soup = BeautifulSoup(html, "lxml")

        # Ищем в пагинации
        pagination = soup.find("div", class_=re.compile("pagination|pages"))
        if pagination:
            links = pagination.find_all("a", href=re.compile(r"page=\d+"))
            pages = []
            for link in links:
                match = re.search(r"page=(\d+)", link.get("href", ""))
                if match:
                    pages.append(int(match.group(1)))
            if pages:
                return max(pages)

        # Или по тексту "из 22193"
        text = soup.get_text()
        match = re.search(r"из\s+(\d+)", text)
        if match:
            total = int(match.group(1))
            return min((total // 30) + 1, 100)  # Ограничим 100

        return 1

    def parse(self):
        """Главный метод парсинга"""
        print(f"🔍 Парсинг города: {self.city}")

        url = f"{self.base_url}/{self.city}/medicamentsalphabetically"
        html = self._get_page(url)

        if not html:
            print("Не удалось получить первую страницу")
            return pd.DataFrame()

        total_pages = self._get_total_pages(html)
        pages_to_parse = min(total_pages, self.max_pages) if self.max_pages else total_pages

        print(f"📄 Всего страниц: {total_pages}, парсим: {pages_to_parse}")

        all_records = []

        for page in range(1, pages_to_parse + 1):
            if page > 1:
                url = f"{self.base_url}/{self.city}/medicamentsalphabetically?page={page}"
                html = self._get_page(url)
                self._delay()

            if html:
                records = self._parse_medicines_from_text(html)
                all_records.extend(records)
                print(f"Страница {page}: +{len(records)} (всего: {len(all_records)})")

        df = pd.DataFrame(all_records)
        print(f"\n Собрано {len(df)} препаратов")
        return df

print("Класс ITekaParser создан")

# ФУНКЦИИ РАБОТЫ С БД

def create_database():
    """Создание базы данных SQLite"""
    conn = sqlite3.connect("medicines.db")
    cursor = conn.cursor()

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS medicines (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT NOT NULL,
            mnn TEXT,
            prescription TEXT,
            pharmacies_count INTEGER,
            city TEXT,
            parsed_at TEXT,
            created_at TEXT DEFAULT CURRENT_TIMESTAMP
        )
    """)

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS parsing_stats (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            city TEXT,
            total_records INTEGER,
            parsed_at TEXT
        )
    """)

    conn.commit()
    conn.close()
    print("База данных создана")

def save_to_database(df):
    """Сохранение DataFrame в БД"""
    if df.empty:
        print("Нет данных для сохранения")
        return

    conn = sqlite3.connect("medicines.db")

    # Очистка
    df_clean = df.copy()
    df_clean["created_at"] = datetime.now().isoformat()
    df_clean = df_clean.drop_duplicates(subset=["name"], keep="first")

    # Сохранение
    df_clean.to_sql("medicines", conn, if_exists="append", index=False)

    # Статистика
    stats = {
        "city": df_clean["city"].iloc[0],
        "total_records": len(df_clean),
        "parsed_at": datetime.now().isoformat()
    }
    pd.DataFrame([stats]).to_sql("parsing_stats", conn, if_exists="append", index=False)

    conn.close()
    print(f"Сохранено {len(df_clean)} записей в medicines.db")

def query_database(sql):
    """Выполнение SQL-запроса"""
    conn = sqlite3.connect("medicines.db")
    result = pd.read_sql_query(sql, conn)
    conn.close()
    return result

print("Функции БД созданы")

# ВИЗУАЛИЗАЦИЯ

def create_charts(df, save_path="charts/"):
    """Создание графиков"""
    os.makedirs(save_path, exist_ok=True)

    # 1. Распределение по аптекам
    plt.figure()
    plt.hist(df["pharmacies_count"], bins=20, color="skyblue", edgecolor="black")
    plt.xlabel("Количество аптек")
    plt.ylabel("Препаратов")
    plt.title("Распределение по доступности")
    plt.tight_layout()
    plt.savefig(f"{save_path}01_distribution.png", dpi=150)
    plt.close()

    # 2. Топ-15
    plt.figure()
    top15 = df.nlargest(15, "pharmacies_count")
    plt.barh(range(len(top15)), top15["pharmacies_count"], color="lightgreen")
    plt.yticks(range(len(top15)), [n[:30] + "..." if len(n) > 30 else n for n in top15["name"]])
    plt.xlabel("Аптек")
    plt.title("Топ-15 самых доступных")
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.savefig(f"{save_path}02_top15.png", dpi=150)
    plt.close()

    # 3. Рецептурный статус
    presc = df["prescription"].value_counts(dropna=False)
    plt.figure()
    plt.pie(presc.values, labels=presc.index, autopct="%1.1f%%",
            colors=["#ff9999", "#66b3ff", "#99ff99"])
    plt.title("Статус отпуска")
    plt.tight_layout()
    plt.savefig(f"{save_path}03_prescription.png", dpi=150)
    plt.close()

    # 4. Топ МНН
    mnn_top = df["mnn"].value_counts().head(10)
    if len(mnn_top) > 0:
        plt.figure()
        plt.bar(range(len(mnn_top)), mnn_top.values, color="coral")
        plt.xticks(range(len(mnn_top)), mnn_top.index, rotation=45, ha="right")
        plt.ylabel("Препаратов")
        plt.title("Топ-10 МНН")
        plt.tight_layout()
        plt.savefig(f"{save_path}04_mnn.png", dpi=150)
        plt.close()

    print(f"Графики сохранены в {save_path}")

def generate_report(df):
    """Генерация отчёта"""
    report = []
    report.append("=" * 50)
    report.append("ОТЧЁТ О ПАРСИНГЕ I-TEKA.KZ")
    report.append(f"Дата: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
    report.append(f"Всего препаратов: {len(df)}")
    report.append(f"Город: {df['city'].iloc[0] if len(df) > 0 else 'N/A'}")
    report.append("")
    report.append(f"Среднее аптек: {df['pharmacies_count'].mean():.1f}")
    report.append(f"Максимум: {df['pharmacies_count'].max()}")
    report.append(f"Минимум: {df['pharmacies_count'].min()}")
    report.append("")
    report.append("--- ПО СТАТУСУ ---")
    for status, count in df["prescription"].value_counts().items():
        report.append(f"  {status}: {count} ({count/len(df)*100:.1f}%)")
    report.append("")
    report.append("--- ТОП-5 ---")
    for _, row in df.nlargest(5, "pharmacies_count").iterrows():
        report.append(f"  • {row['name'][:40]}... ({row['pharmacies_count']} аптек)")

    report_text = "\n".join(report)

    with open("report.txt", "w", encoding="utf-8") as f:
        f.write(report_text)

    print("Отчёт сохранён в report.txt")
    return report_text

print("Функции визуализации созданы")

# ЗАПУСК ПРОГРАММЫ

print("\n" + "=" * 60)
print("  ЗАПУСК ПАРСЕРА I-TEKA.KZ")
print("=" * 60)

# ЭТАП 1: Парсинг
print("\n ЭТАП 1: Парсинг сайта")
parser = ITekaParser(city="astana", max_pages=5)  # Для теста 5 страниц, для полного — None
df = parser.parse()

if not df.empty:
    # ЭТАП 2: База данных
    print("\n ЭТАП 2: Сохранение в SQLite")
    create_database()
    save_to_database(df)

    # Пример SQL-запроса
    print("\n Пример SQL-запроса:")
    sample = query_database("SELECT name, pharmacies_count FROM medicines LIMIT 3")
    print(sample)

    # ЭТАП 3: Визуализация
    print("\n ЭТАП 3: Анализ и визуализация")
    report = generate_report(df)
    print("\n" + report[:300] + "...")
    create_charts(df)

    # Итоги
    print("\n" + "=" * 60)
    print(" ГОТОВО!")
    print(f" Собрано: {len(df)} препаратов")
    print(f" База: medicines.db")
    print(f" Графики: charts/")
    print(f" Отчёт: report.txt")
    print("=" * 60)
else:
    print("❌ Нет данных")

Библиотеки загружены
Класс ITekaParser создан
Функции БД созданы
Функции визуализации созданы

  ЗАПУСК ПАРСЕРА I-TEKA.KZ

 ЭТАП 1: Парсинг сайта
🔍 Парсинг города: astana
📄 Всего страниц: 100, парсим: 5
Страница 1: +25 (всего: 25)
Страница 2: +25 (всего: 50)
Страница 3: +25 (всего: 75)
Страница 4: +25 (всего: 100)
Страница 5: +25 (всего: 125)

 Собрано 125 препаратов

 ЭТАП 2: Сохранение в SQLite
База данных создана
Сохранено 25 записей в medicines.db

 Пример SQL-запроса:
                           name  pharmacies_count
0       Шприц одноразовый 10 мл               606
1        Шприц одноразовый 5 мл               604
2  Мукалтин таблетки 50 мг № 10               602

 ЭТАП 3: Анализ и визуализация
Отчёт сохранён в report.txt

ОТЧЁТ О ПАРСИНГЕ I-TEKA.KZ
Дата: 2026-03-13 17:50
Всего препаратов: 125
Город: astana

Среднее аптек: 564.4
Максимум: 606
Минимум: 546

--- ПО СТАТУСУ ---
  Без рецепта: 80 (64.0%)
  По рецепту: 45 (36.0%)

--- ТОП-5 ---
  • Шприц одноразовый 10 мл....
Графики 